# Lab M2.01: Python for Low Code - Refactoring with AI
## Path 2: Refactor Provided Starter Code

**Step 3, 4, 5: Complete Implementation with Error Handling & Testing**

---

## Story Recap

You have a product description generator that works, but:
- Errors disappear silently
- One function does everything
- Code is repeated and hard to maintain
- No error messages show WHERE problems occur

Your manager says: "If something breaks, I need to know WHAT, WHERE and WHY."

---

## What We'll Build

Step 3: Modularize Functions (break into focused pieces)
Step 4: Add Error Handling (show WHERE and WHY errors occur)
Step 5: Test Everything (verify it all works)

Result: Professional, maintainable, debuggable code

---

# STEP 3 & 4: Complete Refactored Code with Error Handling

## Imports and Setup

In [ ]:
# All imports
import json
import os
from typing import List, Optional, Dict, Any
from pydantic import BaseModel, validator, ValidationError

print("Imports successful!")

## Define Product Model

In [ ]:
class Product(BaseModel):
    """Pydantic model for validating product data."""
    id: str
    name: str
    category: str
    price: float
    features: List[str] = []
    
    @validator('price')
    def price_must_be_positive(cls, v):
        """Price must be positive (greater than 0)."""
        if v <= 0:
            raise ValueError('Price must be positive')
        return v

print("Product model defined!")

---

## REFACTORED CODE: All Helper Functions

### Helper Function #1: Load JSON File (WITH ERROR HANDLING)

In [ ]:
def load_json_file(file_path: str) -> dict:
    """
    Load and parse JSON file with error handling.
    SHOWS WHERE AND WHY errors occur.
    """
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
        return data
    
    except FileNotFoundError:
        error_msg = (
            f"ERROR in load_json_file(): FileNotFoundError\n"
            f"  Location: File '{file_path}' not found\n"
            f"  Current directory: {os.getcwd()}\n"
            f"  Suggestion: Check that the file path is correct and file exists"
        )
        print(error_msg)
        raise
    
    except json.JSONDecodeError as e:
        error_msg = (
            f"ERROR in load_json_file(): JSONDecodeError\n"
            f"  Location: File '{file_path}', line {e.lineno}, column {e.colno}\n"
            f"  Message: {e.msg}\n"
            f"  Suggestion: Check JSON syntax at line {e.lineno}"
        )
        print(error_msg)
        raise

print("Helper #1 defined: load_json_file()")

### Helper Function #2: Validate Product Data (WITH ERROR HANDLING)

In [ ]:
def validate_product_data(product_dict: dict) -> Optional[Product]:
    """
    Validate product data using Pydantic.
    SHOWS WHICH FIELDS are invalid and WHY.
    """
    try:
        product = Product(**product_dict)
        return product
    
    except ValidationError as e:
        product_id = product_dict.get('id', 'unknown')
        error_msg = (
            f"ERROR in validate_product_data(): ValidationError\n"
            f"  Product ID: {product_id}\n"
            f"  Invalid fields:\n"
        )
        for error in e.errors():
            field_name = error['loc'][0]
            error_message = error['msg']
            error_msg += f"    - {field_name}: {error_message}\n"
        error_msg += f"  Suggestion: Fix the invalid fields listed above"
        print(error_msg)
        return None

print("Helper #2 defined: validate_product_data()")

### Helper Function #3: Create Product Prompt

In [ ]:
def create_product_prompt(product: Product) -> str:
    """
    Generate OpenAI prompt for product description.
    Extracted from the loop for reusability.
    """
    features_text = ', '.join(product.features) if product.features else 'No features listed'
    
    prompt = f"""Create a compelling product description for:
Name: {product.name}
Category: {product.category}
Price: ${product.price}
Features: {features_text}

Generate a product description that highlights the key benefits and appeals to customers."""
    
    return prompt

print("Helper #3 defined: create_product_prompt()")

### Helper Function #4: Parse API Response

In [ ]:
def parse_api_response(response) -> str:
    """
    Parse OpenAI API response and extract description.
    Extracted from the loop for testability.
    """
    description = response.choices[0].message.content
    return description

print("Helper #4 defined: parse_api_response()")

### Helper Function #5: Format Output

In [ ]:
def format_output(product: Product, description: str) -> dict:
    """
    Format final output as a dictionary.
    Extracted from the loop for flexibility.
    """
    result = {
        "product_id": product.id,
        "name": product.name,
        "category": product.category,
        "price": product.price,
        "description": description
    }
    return result

print("Helper #5 defined: format_output()")

### Helper Function #6: Save JSON File (WITH ERROR HANDLING)

In [ ]:
def save_json_file(file_path: str, data: dict) -> None:
    """
    Save data to JSON file with error handling.
    SHOWS WHERE save errors occur.
    """
    try:
        with open(file_path, 'w') as f:
            json.dump(data, f, indent=2)
    except IOError as e:
        error_msg = (
            f"ERROR in save_json_file(): IOError\n"
            f"  Location: File '{file_path}'\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Check that directory exists and you have write permissions"
        )
        print(error_msg)
        raise

print("Helper #6 defined: save_json_file()")

---

## STEP 3: Modularize Functions

Now we build modular functions using the helpers.

### Modular Function #1: Load and Validate Products

In [ ]:
def load_and_validate_products(json_path: str) -> List[Product]:
    """
    Load JSON and validate products.
    SINGLE RESPONSIBILITY: Load + Validate
    SHOWS WHERE errors occur
    """
    try:
        # Load JSON file
        data = load_json_file(json_path)
        
        # Validate each product
        products = []
        for item in data.get('products', []):
            product = validate_product_data(item)
            if product:  # Only add if validation passed
                products.append(product)
        
        return products
    
    except Exception as e:
        error_msg = (
            f"ERROR in load_and_validate_products(): {type(e).__name__}\n"
            f"  Location: {json_path}\n"
            f"  Message: {str(e)}"
        )
        print(error_msg)
        raise

print("Modular Function #1 defined: load_and_validate_products()")

### Modular Function #2: Generate Description for One Product

In [ ]:
def generate_description(product: Product, api_client) -> Optional[dict]:
    """
    Generate description for one product using API.
    SINGLE RESPONSIBILITY: Generate description for one product
    SHOWS WHERE API errors occur
    """
    try:
        # Create prompt
        prompt = create_product_prompt(product)
        
        # Call API
        response = api_client.chat.completions.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}]
        )
        
        # Parse response
        description = parse_api_response(response)
        
        # Format output
        result = format_output(product, description)
        
        return result
    
    except Exception as e:
        error_msg = (
            f"ERROR in generate_description(): {type(e).__name__}\n"
            f"  Product: {product.name} (ID: {product.id})\n"
            f"  Message: {str(e)}\n"
            f"  Suggestion: Check API key, rate limits, or try again later"
        )
        print(error_msg)
        return None

print("Modular Function #2 defined: generate_description()")

### Modular Function #3: Process All Products

In [ ]:
def process_products(products: List[Product], api_client) -> List[dict]:
    """
    Process all products and generate descriptions.
    SINGLE RESPONSIBILITY: Orchestrate processing
    """
    results = []
    
    for product in products:
        result = generate_description(product, api_client)
        if result:  # Only add if successful
            results.append(result)
    
    return results

print("Modular Function #3 defined: process_products()")

### Modular Function #4: Main Orchestrator

In [ ]:
def generate_product_descriptions(json_file: str, api_client=None, output_file: str = 'results.json'):
    """
    Main orchestrator: Load -> Validate -> Generate -> Save
    SINGLE RESPONSIBILITY: Orchestrate the entire workflow
    """
    print(f"\n{'='*60}")
    print(f"Processing products from: {json_file}")
    print(f"{'='*60}")
    
    try:
        # Step 1: Load and validate
        print("Step 1: Loading and validating products...")
        products = load_and_validate_products(json_file)
        print(f"  SUCCESS: {len(products)} products validated\n")
        
        if not products:
            print("WARNING: No valid products found!")
            return []
        
        # Step 2: Process products
        print("Step 2: Generating descriptions...")
        results = process_products(products, api_client)
        print(f"  SUCCESS: {len(results)} descriptions generated\n")
        
        # Step 3: Save results
        print("Step 3: Saving results...")
        save_json_file(output_file, {"results": results})
        print(f"  SUCCESS: Results saved to {output_file}\n")
        
        print(f"{'='*60}")
        print(f"COMPLETE: Processed {len(results)} products successfully")
        print(f"{'='*60}\n")
        
        return results
    
    except Exception as e:
        error_msg = (
            f"\nERROR in generate_product_descriptions(): {type(e).__name__}\n"
            f"  Message: {str(e)}\n"
            f"  Process stopped."
        )
        print(error_msg)
        raise

print("Modular Function #4 defined: generate_product_descriptions()")

---

# STEP 5: Test Everything

## Test Setup: Create Test Data

In [ ]:
# Create valid test data
valid_data = {
    "products": [
        {
            "id": "P001",
            "name": "Wireless Bluetooth Headphones",
            "category": "Electronics",
            "price": 99.99,
            "features": ["Bluetooth 5.0", "Noise Cancelling", "30hr Battery"]
        },
        {
            "id": "P002",
            "name": "Smart Watch",
            "category": "Wearables",
            "price": 249.99,
            "features": ["Heart Rate Monitor", "GPS", "Water Resistant"]
        }
    ]
}

with open('test_products_valid.json', 'w') as f:
    json.dump(valid_data, f, indent=2)

print("Test file created: test_products_valid.json")

## Test 1: Load Valid JSON File

In [ ]:
print("\n" + "="*60)
print("TEST 1: Load Valid JSON File")
print("="*60)

try:
    data = load_json_file('test_products_valid.json')
    print("SUCCESS: File loaded correctly!")
    print(f"Number of products: {len(data['products'])}")
    print(f"First product: {data['products'][0]['name']}")
except Exception as e:
    print(f"FAILED: {e}")

## Test 2: Load Missing File (Error Handling)

In [ ]:
print("\n" + "="*60)
print("TEST 2: Load Missing File (Error Handling)")
print("="*60)

try:
    data = load_json_file('nonexistent.json')
except FileNotFoundError:
    print("\nSUCCESS: Error was caught and shown properly!")

## Test 3: Load Invalid JSON (Error Handling)

In [ ]:
# Create invalid JSON
with open('test_invalid.json', 'w') as f:
    f.write('{"products": ["broken json here]}')

print("\n" + "="*60)
print("TEST 3: Load Invalid JSON (Error Handling)")
print("="*60)

try:
    data = load_json_file('test_invalid.json')
except json.JSONDecodeError:
    print("\nSUCCESS: Error was caught and shown properly!")

## Test 4: Validate Valid Product

In [ ]:
print("\n" + "="*60)
print("TEST 4: Validate Valid Product")
print("="*60)

valid_product = {
    "id": "P001",
    "name": "Wireless Headphones",
    "category": "Electronics",
    "price": 99.99,
    "features": ["Bluetooth", "Noise Cancelling"]
}

product = validate_product_data(valid_product)
if product:
    print("SUCCESS: Product validated!")
    print(f"  ID: {product.id}")
    print(f"  Name: {product.name}")
    print(f"  Price: ${product.price}")
else:
    print("FAILED: Validation returned None")

## Test 5: Validate Invalid Product (Negative Price)

In [ ]:
print("\n" + "="*60)
print("TEST 5: Validate Invalid Product (Negative Price)")
print("="*60)

invalid_product = {
    "id": "P002",
    "name": "Smart Watch",
    "category": "Wearables",
    "price": -50.00,  # INVALID
    "features": ["GPS"]
}

product = validate_product_data(invalid_product)
if product is None:
    print("\nSUCCESS: Validation correctly returned None (product is invalid)")

## Test 6: Validate Invalid Product (Missing Required Field)

In [ ]:
print("\n" + "="*60)
print("TEST 6: Validate Invalid Product (Missing Name)")
print("="*60)

invalid_product = {
    "id": "P003",
    # NAME IS MISSING - REQUIRED FIELD
    "category": "Accessories",
    "price": 49.99
}

product = validate_product_data(invalid_product)
if product is None:
    print("\nSUCCESS: Validation correctly returned None (missing required field)")

## Test 7: Create Prompt

In [ ]:
print("\n" + "="*60)
print("TEST 7: Create Prompt")
print("="*60)

test_product = Product(
    id="TEST001",
    name="Test Headphones",
    category="Electronics",
    price=99.99,
    features=["Wireless", "Noise Cancelling"]
)

prompt = create_product_prompt(test_product)
print("Generated Prompt:")
print("-" * 60)
print(prompt)
print("-" * 60)
print("SUCCESS: Prompt created!")

## Test 8: Format Output

In [ ]:
print("\n" + "="*60)
print("TEST 8: Format Output")
print("="*60)

output = format_output(
    product=test_product,
    description="These amazing wireless headphones deliver premium sound quality."
)

print("Formatted Output:")
print("-" * 60)
print(json.dumps(output, indent=2))
print("-" * 60)
print("SUCCESS: Output formatted!")

## Test 9: Load and Validate Products

In [ ]:
print("\n" + "="*60)
print("TEST 9: Load and Validate All Products")
print("="*60)

try:
    products = load_and_validate_products('test_products_valid.json')
    print(f"\nSUCCESS: Loaded and validated {len(products)} products")
    for p in products:
        print(f"  - {p.name} (ID: {p.id})")
except Exception as e:
    print(f"FAILED: {e}")

---

## Test 10: Create Mixed Test Data (Some Valid, Some Invalid)

In [ ]:
# Create test data with MIXED valid and invalid products
mixed_data = {
    "products": [
        {  # VALID
            "id": "P001",
            "name": "Laptop Stand",
            "category": "Accessories",
            "price": 49.99,
            "features": ["Adjustable", "Aluminum"]
        },
        {  # INVALID: negative price
            "id": "P002",
            "name": "Broken Product",
            "category": "Electronics",
            "price": -99.99,
            "features": []
        },
        {  # VALID
            "id": "P003",
            "name": "USB Cable",
            "category": "Accessories",
            "price": 9.99,
            "features": ["3 meters", "Fast charging"]
        },
        {  # INVALID: missing name
            "id": "P004",
            "category": "Electronics",
            "price": 299.99
        }
    ]
}

with open('test_products_mixed.json', 'w') as f:
    json.dump(mixed_data, f, indent=2)

print("Test file created: test_products_mixed.json (2 valid, 2 invalid)")

## Test 11: Load and Validate Mixed Data (Shows Error Handling)

In [ ]:
print("\n" + "="*60)
print("TEST 11: Load and Validate Mixed Data")
print("(Shows error handling for invalid products)")
print("="*60)

try:
    products = load_and_validate_products('test_products_mixed.json')
    print(f"\nRESULT: {len(products)} valid products (2 invalid products were skipped)")
    print("\nValid products:")
    for p in products:
        print(f"  - {p.name} (ID: {p.id}, Price: ${p.price})")
except Exception as e:
    print(f"FAILED: {e}")

---

## Summary of Tests

In [ ]:
print("\n" + "="*60)
print("TESTING SUMMARY")
print("="*60)

summary = """
TEST RESULTS:

✓ TEST 1: Load valid JSON - SUCCESS
✓ TEST 2: Load missing file - ERROR SHOWN (FileNotFoundError)
✓ TEST 3: Load invalid JSON - ERROR SHOWN (JSONDecodeError)
✓ TEST 4: Validate valid product - SUCCESS
✓ TEST 5: Validate negative price - ERROR SHOWN (validation error)
✓ TEST 6: Validate missing field - ERROR SHOWN (validation error)
✓ TEST 7: Create prompt - SUCCESS
✓ TEST 8: Format output - SUCCESS
✓ TEST 9: Load and validate all - SUCCESS
✓ TEST 11: Mixed valid/invalid - SHOWS ERROR HANDLING

KEY IMPROVEMENTS:

1. BEFORE: Silent failures (except: pass)
   AFTER: Clear error messages show WHERE and WHY

2. BEFORE: One function does everything
   AFTER: 6 helper functions + 4 modular functions (single responsibility)

3. BEFORE: Code repeated in multiple places
   AFTER: Each piece extracted once, reusable

4. BEFORE: Hard to test
   AFTER: Each function can be tested independently

5. BEFORE: Hard to modify
   AFTER: Change one function without affecting others
"""

print(summary)